## Install Required Libraries 

In [1]:
!pip install transformers datasets torch pandas scikit-learn

### Import Required Libraries

In [2]:
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer
import tensorflow as tf



from transformers import BertTokenizer, BertForSequenceClassification

from torch.optim import AdamW

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

2026-03-12 19:13:29.187342: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773342809.385270      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773342809.442763      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773342809.931579      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773342809.931612      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773342809.931615      55 computation_placer.cc:177] computation placer alr

### Load dataset

In [3]:
DATA_PATH = "/kaggle/input/datasets/mrsikandarali/metahate-dataset/your_dataset_updated.tsv"

df = pd.read_csv(DATA_PATH, sep="\t")

print(df.head())
print("Dataset shape:", df.shape)

   label                                               text
0      0  !!! RT @mayasolovely: As a woman you shouldn't...
1      0  !!!!! RT @mleew17: boy dats cold...tyga dwn ba...
2      0  !!!!!!! RT @UrKindOfBrand Dawg!!!! RT @80sbaby...
3      0  !!!!!!!!! RT @C_G_Anderson: @viva_based she lo...
4      0  !!!!!!!!!!!!! RT @ShenikaRoberts: The shit you...
Dataset shape: (1101165, 2)


### Split the data for train test

In [5]:
train_texts, test_texts, train_labels, test_labels = train_test_split(
    df['text'],
    df['label'],
    test_size=0.2,
    random_state=42
)

# detect and init the TPU

In [6]:
# Detect hardware, return appropriate distribution strategy
# Step 1: Import TensorFlow

try:
    # TPU detection
    tpu = tf.distribute.cluster_resolver.TPUClusterResolver()
    print('Running on TPU:', tpu.master())
except ValueError:
    tpu = None

if tpu:
    # Connect to TPU
    tf.config.experimental_connect_to_cluster(tpu)
    tf.tpu.experimental.initialize_tpu_system(tpu)
    strategy = tf.distribute.experimental.TPUStrategy(tpu)
else:
    # Default strategy for CPU and GPU
    strategy = tf.distribute.get_strategy()

print("REPLICAS:", strategy.num_replicas_in_sync)

# Check available devices

print("GPU Available:", tf.config.list_physical_devices('GPU') != [])
print("CPU Available:", tf.config.list_physical_devices('CPU') != [])
print("TPU Available:", tf.config.list_physical_devices('TPU') != [])

REPLICAS: 1
GPU Available: True
CPU Available: True
TPU Available: False


### Load BERT Tokenizer 
Tokenizer converts text into numbers for BERT

In [7]:
# Load fast tokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased", use_fast=True)

MAX_LEN = 128


def tokenize_batch(texts, batch_size=50000, max_length=MAX_LEN):

    encodings = {
        "input_ids": [],
        "attention_mask": [],
        "token_type_ids": []
    }

    total = len(texts)

    for i in range(0, total, batch_size):

        batch = texts[i:i+batch_size].tolist()

        tokens = tokenizer(
            batch,
            truncation=True,
            padding="max_length",
            max_length=max_length
        )

        encodings["input_ids"].extend(tokens["input_ids"])
        encodings["attention_mask"].extend(tokens["attention_mask"])
        encodings["token_type_ids"].extend(tokens["token_type_ids"])

        print(f"Processed {min(i+batch_size, total)} / {total}")

    return encodings


# Tokenize datasets
train_encodings = tokenize_batch(train_texts)
test_encodings = tokenize_batch(test_texts)


# Save encoded dataset
torch.save({
    "train_encodings": train_encodings,
    "test_encodings": test_encodings,
    "train_labels": train_labels,
    "test_labels": test_labels
}, "dataset_encodings.pt")


# Save tokenizer
tokenizer.save_pretrained("saved_tokenizer")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Processed 50000 / 880932
Processed 100000 / 880932
Processed 150000 / 880932
Processed 200000 / 880932
Processed 250000 / 880932
Processed 300000 / 880932
Processed 350000 / 880932
Processed 400000 / 880932
Processed 450000 / 880932
Processed 500000 / 880932
Processed 550000 / 880932
Processed 600000 / 880932
Processed 650000 / 880932
Processed 700000 / 880932
Processed 750000 / 880932
Processed 800000 / 880932
Processed 850000 / 880932
Processed 880932 / 880932
Processed 50000 / 220233
Processed 100000 / 220233
Processed 150000 / 220233
Processed 200000 / 220233
Processed 220233 / 220233


('saved_tokenizer/tokenizer_config.json', 'saved_tokenizer/tokenizer.json')

# Load tokenize dataset


In [8]:
# Load dataset
data = torch.load("dataset_encodings.pt", map_location="cpu", weights_only=False)

# Extract items
train_encodings = data["train_encodings"]
test_encodings = data["test_encodings"]
train_labels = data["train_labels"]
test_labels = data["test_labels"]

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained("saved_tokenizer")

print("Dataset loaded successfully")

Dataset loaded successfully


### Create PyTorch Dataset Class

In [9]:
class HateDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels
    
    def __len__(self):
        return len(self.labels)
    
    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        # Use iloc to get by position
        item["labels"] = torch.tensor(self.labels.iloc[idx])
        return item

### Create Train and Test Dataset

In [10]:
train_dataset = HateDataset(train_encodings, train_labels)
test_dataset = HateDataset(test_encodings, test_labels)

### Create DataLoaders

In [11]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32)

### Load BERT Model for Classification

In [12]:
from transformers import BertForSequenceClassification
import torch

# Load BERT for sequence classification with 2 labels
model = BertForSequenceClassification.from_pretrained(
    'bert-base-uncased',
    num_labels=2
)

# Choose device: GPU if available, otherwise CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Move the model to the device
model.to(device)

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

### Define Optimizer

In [14]:
!pip install tqdm 
from tqdm import tqdm
optimizer = AdamW(model.parameters(), lr=2e-5)

### Training the BERT Model

In [15]:
# epochs = 5   # Increased number of epochs

# model.train()

# for epoch in range(epochs):
    
#     total_loss = 0
    
#     print(f"\nStarting Epoch {epoch+1}/{epochs}")
    
#     progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}", leave=True)
    
#     for batch in progress_bar:
        
#         optimizer.zero_grad()
        
#         input_ids = batch['input_ids'].to(device)
#         attention_mask = batch['attention_mask'].to(device)
#         labels = batch['labels'].to(device)
        
#         outputs = model(
#             input_ids=input_ids,
#             attention_mask=attention_mask,
#             labels=labels
#         )
        
#         loss = outputs.loss
#         total_loss += loss.item()
        
#         loss.backward()
#         optimizer.step()
        
#         # Update progress bar with current loss
#         progress_bar.set_postfix({"Batch Loss": loss.item()})
    
#     avg_loss = total_loss / len(train_loader)
    
#     print(f"Epoch {epoch+1} Completed | Average Loss: {avg_loss:.4f}")

In [16]:
from torch.cuda.amp import autocast, GradScaler
from sklearn.metrics import accuracy_score

# Mixed precision scaler (speed improvement)
scaler = GradScaler()

epochs = 5
patience = 2          # Early stopping patience
best_val_loss = float("inf")
counter = 0

for epoch in range(epochs):

    # ========================
    # TRAINING
    # ========================
    model.train()
    total_loss = 0

    print(f"\nStarting Epoch {epoch+1}/{epochs}")

    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}")

    for batch in progress_bar:

        optimizer.zero_grad()

        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        # Mixed precision (faster training)
        with autocast():

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )

            loss = outputs.loss

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()

        progress_bar.set_postfix({"Batch Loss": loss.item()})

    avg_train_loss = total_loss / len(train_loader)

    # ========================
    # VALIDATION
    # ========================
    model.eval()

    val_loss = 0
    predictions = []
    true_labels = []

    with torch.no_grad():

        for batch in test_loader:

            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )

            loss = outputs.loss
            logits = outputs.logits

            val_loss += loss.item()

            preds = torch.argmax(logits, dim=1)

            predictions.extend(preds.cpu().numpy())
            true_labels.extend(labels.cpu().numpy())

    avg_val_loss = val_loss / len(test_loader)
    accuracy = accuracy_score(true_labels, predictions)

    print(f"Epoch {epoch+1}")
    print(f"Train Loss: {avg_train_loss:.4f}")
    print(f"Validation Loss: {avg_val_loss:.4f}")
    print(f"Validation Accuracy: {accuracy:.4f}")

    # ========================
    # EARLY STOPPING
    # ========================
    if avg_val_loss < best_val_loss:

        best_val_loss = avg_val_loss
        counter = 0

        torch.save(model.state_dict(), "best_bert_model.pt")

        print("Model improved. Saved.")

    else:

        counter += 1
        print(f"No improvement. Early stop counter: {counter}/{patience}")

        if counter >= patience:

            print("Early stopping triggered.")
            break

/tmp/ipykernel_55/1515397389.py:5: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()



Starting Epoch 1/5


Epoch 1:   0%|          | 0/27530 [00:00<?, ?it/s]/tmp/ipykernel_55/1515397389.py:33: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
Epoch 1: 100%|██████████| 27530/27530 [1:31:05<00:00,  5.04it/s, Batch Loss=0.223] 


Epoch 1
Train Loss: 0.2503
Validation Loss: 0.2276
Validation Accuracy: 0.9011
Model improved. Saved.

Starting Epoch 2/5


Epoch 2:   0%|          | 0/27530 [00:00<?, ?it/s]/tmp/ipykernel_55/1515397389.py:33: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
Epoch 2: 100%|██████████| 27530/27530 [1:31:13<00:00,  5.03it/s, Batch Loss=0.305] 


Epoch 2
Train Loss: 0.1992
Validation Loss: 0.2208
Validation Accuracy: 0.9059
Model improved. Saved.

Starting Epoch 3/5


Epoch 3:   0%|          | 0/27530 [00:00<?, ?it/s]/tmp/ipykernel_55/1515397389.py:33: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
Epoch 3: 100%|██████████| 27530/27530 [1:31:12<00:00,  5.03it/s, Batch Loss=0.0101]


Epoch 3
Train Loss: 0.1585
Validation Loss: 0.2418
Validation Accuracy: 0.9070
No improvement. Early stop counter: 1/2

Starting Epoch 4/5


Epoch 4:   0%|          | 0/27530 [00:00<?, ?it/s]/tmp/ipykernel_55/1515397389.py:33: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
Epoch 4: 100%|██████████| 27530/27530 [1:30:52<00:00,  5.05it/s, Batch Loss=0.000238]


Epoch 4
Train Loss: 0.1231
Validation Loss: 0.2627
Validation Accuracy: 0.9033
No improvement. Early stop counter: 2/2
Early stopping triggered.


## Save as folder

In [17]:
# # Save trained BERT model
# model.save_pretrained("bert_hate_speech_model")

# # Save tokenizer as well
# tokenizer.save_pretrained("bert_hate_speech_model")

# print("Model and tokenizer saved successfully!")

In [18]:
# # Save
model.save_pretrained("./best_bert_model")
tokenizer.save_pretrained("./best_bert_model")  # save tokenizer too
print("Model and tokenizer saved successfully!")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model and tokenizer saved successfully!


In [19]:
# Load later
from transformers import BertForSequenceClassification, BertTokenizer

model = BertForSequenceClassification.from_pretrained("./best_bert_model")
tokenizer = BertTokenizer.from_pretrained("./best_bert_model")
model.to(device)
model.eval()

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

### Model Evaluation

### Accuracy and Classification Report

In [20]:
# model.eval()

predictions = []
true_labels = []

with torch.no_grad():
    
    for batch in test_loader:
        
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        
        outputs = model(
            input_ids,
            attention_mask=attention_mask
        )
        
        logits = outputs.logits
        preds = torch.argmax(logits, dim=1)
        
        predictions.extend(preds.cpu().numpy())
        true_labels.extend(labels.cpu().numpy())

In [21]:
accuracy = accuracy_score(true_labels, predictions)

print("Accuracy:", accuracy)

print("\nClassification Report:\n")
print(classification_report(true_labels, predictions))

Accuracy: 0.9033160334736393

Classification Report:

              precision    recall  f1-score   support

           0       0.94      0.94      0.94    173537
           1       0.78      0.76      0.77     46696

    accuracy                           0.90    220233
   macro avg       0.86      0.85      0.85    220233
weighted avg       0.90      0.90      0.90    220233



### Predict Hate Speech for New Text

In [22]:
def predict(text):
    
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )
    
    inputs = {key: val.to(device) for key, val in inputs.items()}
    
    outputs = model(**inputs)
    
    prediction = torch.argmax(outputs.logits, dim=1).item()
    
    return prediction


example = "I hate this community"

print("Prediction:", predict(example))

Prediction: 0
